# Ch.3 — Evaluation Metrics for Classification

**FaceAI**: Why accuracy lies for imbalanced classes (Bald 2.5%, Mustache 4.2%).

**Goal**: Confusion matrix, precision/recall, F1, ROC-AUC, PR-AUC, multi-label metrics.

**Key insight**: 97.5% accuracy on Bald by always predicting "Not Bald" — the accuracy paradox.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import numpy, matplotlib.pyplot, Path
# 2. Import make_classification, train_test_split, StratifiedKFold, cross_val_score
# 3. Import StandardScaler, LogisticRegression
# 4. Import classification_report, confusion_matrix, ConfusionMatrixDisplay,
#    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
#    f1_score, precision_score, recall_score from sklearn.metrics
# 5. Set IMG_DIR, SAVE_KW, np.random.seed(42)
#
# Hint:
#   from sklearn.model_selection import StratifiedKFold, cross_val_score
#   from sklearn.metrics import (precision_recall_curve, average_precision_score,
#                                f1_score, precision_score, recall_score)


## §0 Data — Three Attributes with Different Imbalance

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define helper make_attr(pos_rate, name, n=5000) that calls
#    make_classification(weights=[1-pos_rate, pos_rate], flip_y=0.03, random_state=42)
# 2. Build datasets dict with 3 entries:
#    'Smiling (48%)'    -> make_attr(0.48, 'Smiling')
#    'Eyeglasses (13%)' -> make_attr(0.13, 'Eyeglasses')
#    'Bald (2.5%)'      -> make_attr(0.025, 'Bald')
# 3. Print positive count and rate for each attribute
#
# Hint:
#   def make_attr(pos_rate, name, n=5000):
#       X, y = make_classification(n_samples=n, n_features=200, n_informative=30,
#                                  n_redundant=20, weights=[1-pos_rate, pos_rate],
#                                  flip_y=0.03, random_state=42)
#       return X, y, name
#   datasets = {'Smiling (48%)': make_attr(???, 'Smiling'), ...}


## §1 The Accuracy Paradox

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Extract X_bald, y_bald from datasets['Bald (2.5%)']; split and scale
# 2. Model A: y_pred_always_no = np.zeros_like(y_te)
#    compute accuracy and f1_score(zero_division=0)
# 3. Model B: train LogisticRegression(class_weight='balanced');
#    compute accuracy and f1_score on its predictions
# 4. Print both rows showing the paradox:
#    'dumb' model has higher accuracy but zero F1
#
# Hint:
#   y_pred_always_no = np.zeros_like(y_te)
#   acc_always_no = (y_pred_always_no == y_te).mean()
#   f1_always_no  = f1_score(y_te, y_pred_always_no, zero_division=0)
#   lr = LogisticRegression(C=1.0, max_iter=500, class_weight='balanced', random_state=42)
#   print("  Always Not-Bald: accuracy=..., F1=...")


## §2 Confusion Matrices Across Attributes

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# 2. For each (ax, dataset): split + scale + fit LogisticRegression(C=1.0)
# 3. Call ConfusionMatrixDisplay.from_estimator(lr, X_te_s, y_te,
#    display_labels=[f'Not {label}', label], cmap='Blues', ax=ax)
# 4. Set per-subplot title; add supertitle; save to 'confusion_matrices.png'
#
# Hint:
#   fig, axes = plt.subplots(1, 3, figsize=(15, 4))
#   for ax, (name, (X, y, label)) in zip(axes, datasets.items()):
#       # split, scale, fit lr ...
#       ConfusionMatrixDisplay.from_estimator(
#           lr, X_te_s, y_te,
#           display_labels=[f'Not {label}', label], cmap='Blues', ax=ax)


## §3 ROC Curves

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create one figure; loop over datasets with paired colors
# 2. For each: split + scale + fit LogReg + y_prob = predict_proba[:, 1]
# 3. Compute fpr, tpr, _ = roc_curve(y_te, y_prob) and auc = roc_auc_score(...)
# 4. Plot each curve with name + AUC in legend; add k-- diagonal baseline
# 5. Save to IMG_DIR / 'roc_curves.png'
#
# Hint:
#   colors = ['blue', 'green', 'red']
#   for color, (name, (X, y, _)) in zip(colors, datasets.items()):
#       # split, scale, fit lr, get y_prob ...
#       fpr, tpr, _ = roc_curve(y_te, y_prob)
#       auc = roc_auc_score(y_te, y_prob)
#       ax.plot(fpr, tpr, color=color, label=f'{name} (AUC={auc:.3f})')


## §4 PR Curves (Better for Imbalanced Data)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create one figure; loop over datasets with paired colors
# 2. For each: fit LogReg, compute
#    prec, rec, _ = precision_recall_curve(y_te, y_prob)
#    ap = average_precision_score(y_te, y_prob)
# 3. Plot recall on x-axis, precision on y-axis (axes are swapped vs ROC)
# 4. Save to IMG_DIR / 'pr_curves.png'
#
# Hint:
#   prec, rec, _ = precision_recall_curve(y_te, y_prob)
#   ap = average_precision_score(???, ???)
#   ax.plot(rec, prec, color=color, label=f'{name} (AP={ap:.3f})')


## §5 Threshold Tuning for Bald (2.5%)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Extract Bald data; split, scale, fit LogReg; get y_prob
# 2. Define thresholds = np.arange(0.02, 0.8, 0.02)
# 3. Loop: for each t compute f1, precision, recall (use zero_division=0)
# 4. Plot all three vs threshold; mark optimal t and default 0.5
# 5. Print F1 at t=0.5 vs optimal t; save to 'bald_threshold_tuning.png'
#
# Hint:
#   thresholds = np.arange(0.02, 0.8, 0.02)
#   f1s = [f1_score(y_te, (y_prob >= t).astype(int), zero_division=0)
#          for t in thresholds]
#   best_t = thresholds[np.argmax(f1s)]
#   ax.axvline(best_t, color='blue', linestyle=':')
#   ax.axvline(0.5, color='gray', linestyle='--', label='Default 0.5')


## §6 Multi-Label Metrics Preview

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Simulate y_true_multi with np.random.binomial(1, 0.3, size=(n_samples, n_attrs))
# 2. Create y_pred_multi by flipping ~10% of labels:
#    flip_mask = np.random.random(size=(n_samples, n_attrs)) < 0.10
# 3. Compute Hamming Loss:     (y_true != y_pred).mean()
# 4. Compute Subset Accuracy:  (y_true == y_pred).all(axis=1).mean()
# 5. Compute per-attribute accuracy along axis=0; print all three metrics
#
# Hint:
#   y_true_multi = np.random.binomial(1, 0.3, size=(n_samples, n_attrs))
#   flip_mask    = np.random.random(size=(n_samples, n_attrs)) < 0.10
#   y_pred_multi = y_true_multi.copy()
#   y_pred_multi[flip_mask] = 1 - y_pred_multi[flip_mask]
#   hamming    = (y_true_multi != y_pred_multi).mean()
#   subset_acc = (y_true_multi == y_pred_multi).all(axis=1).mean()


## §7 Cross-Validation Stability

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Extract Smiling data; scale with StandardScaler
# 2. Create StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# 3. Call cross_val_score() three times: scoring='accuracy', 'f1', 'roc_auc'
# 4. Print mean ± std for each metric
# 5. Plot bar chart with error bars (yerr=stds, capsize=5); save to 'cv_stability.png'
#
# Hint:
#   cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#   scores_acc = cross_val_score(lr, X_s, y_smile, cv=cv, scoring='accuracy')
#   scores_f1  = cross_val_score(lr, X_s, y_smile, cv=cv, scoring='f1')
#   scores_auc = cross_val_score(lr, X_s, y_smile, cv=cv, scoring='roc_auc')
#   ax.bar(metrics, means, yerr=stds, capsize=5)


## §8 Summary

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Print separator line and chapter title
# 2. Print 5 key lessons: accuracy paradox, F1/PR-AUC, threshold, multi-label, CV
# 3. Print ACCURACY constraint validation note
#
# Hint:
#   print("Key Lessons:")
#   print("  1. Accuracy lies for imbalanced classes (Bald: 97.5% by doing nothing)")
#   print("  2. F1, PR-AUC are essential for rare attributes")
#   print("  3. Threshold tuning is critical (0.5 default is often wrong)")
#   print("  4. Multi-label: Hamming loss vs subset accuracy")
#   print("  5. Cross-validation gives confidence intervals")


## Exercises

1. **Calibration curve**: Plot the calibration curve for LogReg on Smiling. Is it well-calibrated?
2. **Matthews Correlation Coefficient**: Compute MCC for Bald. Compare with F1 — which is more informative?
3. **Multi-label heatmap**: Simulate 40 attributes, compute per-attribute F1, plot as a sorted bar chart.

In [ ]:
# Exercise 1: Calibration curve
# TODO: Implement this cell
#
# Hint: from sklearn.calibration import calibration_curve;
#   frac_pos, mean_pred = calibration_curve(y_te, y_prob, n_bins=10);
#   plot mean_pred (x) vs frac_pos (y); compare with perfect-calibration diagonal


In [ ]:
# Exercise 2: Matthews Correlation Coefficient
# TODO: Implement this cell
#
# Hint: from sklearn.metrics import matthews_corrcoef;
#   mcc = matthews_corrcoef(y_te, y_pred) for the Bald model;
#   compare with f1_score — MCC handles class imbalance more symmetrically


In [ ]:
# Exercise 3: Multi-label F1 heatmap
# TODO: Implement this cell
#
# Hint: simulate 40 binary attributes with varying pos_rate;
#   for each attribute train LogReg and compute f1_score;
#   sort attributes by F1 and plot as a sorted ax.bar() chart
